In [10]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [11]:
from pathlib import Path

# -----------------------
# File paths
# -----------------------

base_dir = Path(
    "RFID-ExSim-dataset/rfid_dataset/rfid_dataset/data/processed"
)

paths = {
    "S1": base_dir / "rfid_dataset_S1_all.jsonl",
    "S2": base_dir / "rfid_dataset_S2_all.jsonl",
    "S3": base_dir / "rfid_dataset_S3_all.jsonl",
    "S4": base_dir / "rfid_dataset_S4_all.jsonl",
    "S5": base_dir / "rfid_dataset_S5_all.jsonl",
}

# Confirm files exist
for scenario, path in paths.items():
    print(scenario, path, "exists:", path.exists())

# -----------------------
# Load and merge datasets
# -----------------------

dfs = []

for scenario, path in paths.items():
    temp = pd.read_json(path, lines=True)

    # Keep scenario from filename as reliable label
    temp["scenario"] = scenario

    # S1 = normal, S2-S5 = anomaly
    temp["label"] = 0 if scenario == "S1" else 1

    dfs.append(temp)

df_all = pd.concat(dfs, ignore_index=True)

print("Full dataset shape:", df_all.shape)
print(df_all["scenario"].value_counts())
print(df_all.columns)

df_all.head()

S1 RFID-ExSim-dataset/rfid_dataset/rfid_dataset/data/processed/rfid_dataset_S1_all.jsonl exists: True
S2 RFID-ExSim-dataset/rfid_dataset/rfid_dataset/data/processed/rfid_dataset_S2_all.jsonl exists: True
S3 RFID-ExSim-dataset/rfid_dataset/rfid_dataset/data/processed/rfid_dataset_S3_all.jsonl exists: True
S4 RFID-ExSim-dataset/rfid_dataset/rfid_dataset/data/processed/rfid_dataset_S4_all.jsonl exists: True
S5 RFID-ExSim-dataset/rfid_dataset/rfid_dataset/data/processed/rfid_dataset_S5_all.jsonl exists: True
Full dataset shape: (408422, 15)
scenario
S5    289523
S4     41719
S3     32400
S2     31776
S1     13004
Name: count, dtype: int64
Index(['timestamp_iso', 'session_id', 'scenario', 'device_id', 'port',
       'tag_local_id', 'uid_hash', 'uid_plain_local', 'distance_cm',
       'orientation_deg', 'event', 'raw_payload', 'label', 'reader',
       'source_file'],
      dtype='object')


,timestamp_iso,session_id,scenario,device_id,port,tag_local_id,uid_hash,uid_plain_local,distance_cm,orientation_deg,event,raw_payload,label,reader,source_file
0,2025-11-04 08:12:55.469573+00:00,S20251104_S1_TAG01_d1_o0_run1,S1,ESP32_B,COM4,TAG01,9720a7f2733d,83C11D13,1.0,0.0,read_success,UID: 83 C1 1D 13,0,NaN,NaN
1,2025-11-04 08:13:09.703865+00:00,S20251104_S1_TAG01_d1_o0_run1,S1,ESP32_B,COM4,TAG01,9720a7f2733d,83C11D13,1.0,0.0,read_success,UID: 83 C1 1D 13,0,NaN,NaN
2,2025-11-04 08:13:15.142252+00:00,S20251104_S1_TAG01_d1_o0_run1,S1,ESP32_B,COM4,TAG01,9720a7f2733d,83C11D13,1.0,0.0,read_success,UID: 83 C1 1D 13,0,NaN,NaN
3,2025-11-04 08:13:16.930616+00:00,S20251104_S1_TAG01_d1_o0_run1,S1,ESP32_B,COM4,TAG01,9720a7f2733d,83C11D13,1.0,0.0,read_success,UID: 83 C1 1D 13,0,NaN,NaN
4,2025-11-04 08:13:20.141495+00:00,S20251104_S1_TAG01_d1_o0_run1,S1,ESP32_B,COM4,TAG01,9720a7f2733d,83C11D13,1.0,0.0,read_success,UID: 83 C1 1D 13,0,NaN,NaN


In [12]:
# -----------------------
# Train on S1 only
# -----------------------

train_df = df_all[df_all["scenario"] == "S1"].copy()

X_train = train_df[features]

# Scale using only normal training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train Isolation Forest
model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

model.fit(X_train_scaled)

print("Training complete.")
print("Training rows:", X_train.shape[0])

KeyError: "['delta_t', 'device_id_enc', 'tag_id_enc'] not in index"

In [ ]:
# -----------------------
# Test on merged S1-S5
# -----------------------

test_df = df_all.copy()

X_test = test_df[features]
y_test = test_df["label"]

# Use the same scaler fitted on S1
X_test_scaled = scaler.transform(X_test)

# Predict
raw_preds = model.predict(X_test_scaled)

# Isolation Forest output:
#  1 = normal
# -1 = anomaly
#
# Convert to:
# 0 = normal
# 1 = anomaly
test_df["pred"] = np.where(raw_preds == -1, 1, 0)

# Higher score = more anomalous
test_df["anomaly_score"] = -model.decision_function(X_test_scaled)

# -----------------------
# Overall evaluation
# -----------------------

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_df["pred"]))

print("\nClassification Report:")
print(classification_report(y_test, test_df["pred"]))

# -----------------------
# Scenario-level results
# -----------------------

scenario_summary = test_df.groupby("scenario").agg(
    total=("pred", "count"),
    true_anomaly_rate=("label", "mean"),
    predicted_anomalies=("pred", "sum"),
    predicted_anomaly_rate=("pred", "mean"),
    avg_anomaly_score=("anomaly_score", "mean")
)

scenario_summary